In [ ]:
# ==================== CELLULE D'ENTRAÎNEMENT ====================
# Exécutez CECI avant d'essayer de sauvegarder

import torch
import torch.nn as nn
import torch.optim as optim
import random

# Paramètres
hidden_size = 256
learning_rate = 0.005
n_iters = 50000  # Vous pouvez réduire à 20000 pour aller plus vite
print_every = 5000

# Définition des modèles
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
    
    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output, hidden = self.gru(embedded, hidden)
        return output, hidden
    
    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)
    
    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output, hidden = self.gru(embedded, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden

# Initialisation
encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()

# Fonction d'entraînement
def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion):
    encoder_hidden = encoder.initHidden()
    
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()
    
    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)
    
    loss = 0
    
    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
    
    decoder_input = torch.tensor([[SOS_token]], device=device)
    decoder_hidden = encoder_hidden
    
    for di in range(target_length):
        decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
        loss += criterion(decoder_output, target_tensor[di])
        decoder_input = target_tensor[di]
    
    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()
    
    return loss.item() / target_length

# Fonction pour convertir une phrase en tenseur
def indexesFromSentence(lang, sentence):
    indexes = []
    for word in sentence.split(' '):
        if word in lang.word2index:
            indexes.append(lang.word2index[word])
        else:
            indexes.append(UNK_token)
    return indexes

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)

# Entraînement
print("=" * 50)
print("DÉBUT DE L'ENTRAÎNEMENT Seq2Seq")
print("=" * 50)
print(f"Device: {device}")
print(f"Hidden size: {hidden_size}")
print(f"Itérations: {n_iters}")
print(f"Paires disponibles: {len(pairs)}")
print("=" * 50)

losses = []

for iter in range(1, n_iters + 1):
    # Choisir une paire aléatoire
    training_pair = random.choice(pairs)
    input_tensor = tensorFromSentence(input_lang, training_pair[0])
    target_tensor = tensorFromSentence(output_lang, training_pair[1])
    
    loss = train(input_tensor, target_tensor, encoder, decoder,
                 encoder_optimizer, decoder_optimizer, criterion)
    
    losses.append(loss)
    
    if iter % print_every == 0:
        avg_loss = sum(losses[-print_every:]) / print_every
        print(f"Itération {iter} ({iter/n_iters*100:.1f}%) - Loss moyenne: {avg_loss:.4f}")

print("\n" + "=" * 50)
print("ENTRAÎNEMENT TERMINÉ !")
print("=" * 50)

# ==================== SAUVEGARDE DU MODÈLE ====================
torch.save({
    'encoder_state': encoder.state_dict(),
    'decoder_state': decoder.state_dict(),
    'input_lang': input_lang,
    'output_lang': output_lang,
    'hidden_size': hidden_size,
    'losses': losses
}, 'seq2seq_model.pth')

print("✅ Modèle sauvegardé dans 'seq2seq_model.pth'")

DÉBUT DE L'ENTRAÎNEMENT Seq2Seq
Device: cpu
Hidden size: 256
Itérations: 50000
Paires disponibles: 10000
Itération 5000 (10.0%) - Loss moyenne: 3.6460


In [ ]:
# -*- coding: utf-8 -*-
# partie3_seq2seq_tatoeba.ipynb

# %% [markdown]
# # PARTIE III - Seq2Seq : Traduction Français → Anglais sur Tatoeba
# ## Deep Learning - EMSI 2025-2026

# %% [markdown]
# ### 1. Import des bibliothèques

# %%
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import numpy as np
import matplotlib.pyplot as plt
import unicodedata
import re
from io import open
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# %% [markdown]
# ### 2. Préparation des données Tatoeba

# %%
# Fonctions de prétraitement
SOS_token = 0
EOS_token = 1
UNK_token = 2
MAX_LENGTH = 20

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {"SOS": SOS_token, "EOS": EOS_token, "UNK": UNK_token}
        self.word2count = {}
        self.index2word = {SOS_token: "SOS", EOS_token: "EOS", UNK_token: "UNK"}
        self.n_words = 3
    
    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)
    
    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

# Normalisation Unicode
def unicodeToAscii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s

# %% [markdown]
# ### 3. Création du dataset Tatoeba

# %%
# IMPORTANT: Téléchargez d'abord le fichier depuis:
# https://www.manythings.org/anki/fra-eng.zip
# Extrayez et placez 'fra.txt' dans data/tatoeba/

def readLangs(lang1, lang2, reverse=False):
    print("Lecture des lignes...")
    
    # Chemin du fichier (à adapter selon où vous avez mis le fichier)
    file_path = 'data/tatoeba/fra.txt'
    
    lines = open(file_path, encoding='utf-8').read().strip().split('\n')
    
    pairs = [[normalizeString(s) for s in l.split('\t')[:2]] for l in lines]
    
    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)
    
    return input_lang, output_lang, pairs

def filterPair(p, max_length=MAX_LENGTH):
    return len(p[0].split(' ')) < max_length and len(p[1].split(' ')) < max_length

def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

def prepareData(lang1, lang2, reverse=False, max_pairs=10000):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print(f"Total pairs: {len(pairs)}")
    
    pairs = filterPairs(pairs)
    print(f"Pairs filtrés: {len(pairs)}")
    
    # Limiter pour l'entraînement
    pairs = pairs[:max_pairs]
    
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    
    print(f"Taille vocabulaire input ({input_lang.name}): {input_lang.n_words}")
    print(f"Taille vocabulaire output ({output_lang.name}): {output_lang.n_words}")
    
    return input_lang, output_lang, pairs

# Préparation des données (français -> anglais)
input_lang, output_lang, pairs = prepareData('fra', 'eng', reverse=False, max_pairs=10000)
print(f"\nQuelques exemples de paires:")
for i in range(5):
    print(f"{pairs[i][0]} -> {pairs[i][1]}")

# %% [markdown]
# ### 4. Conversion en tenseurs

# %%
def indexesFromSentence(lang, sentence):
    indexes = []
    for word in sentence.split(' '):
        if word in lang.word2index:
            indexes.append(lang.word2index[word])
        else:
            indexes.append(UNK_token)
    return indexes

def tensorFromSentence(lang, sentence, device):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)

def tensorFromPair(pair, input_lang, output_lang, device):
    input_tensor = tensorFromSentence(input_lang, pair[0], device)
    target_tensor = tensorFromSentence(output_lang, pair[1], device)
    return input_tensor, target_tensor

class TranslationDataset(Dataset):
    def __init__(self, pairs, input_lang, output_lang, device):
        self.pairs = pairs
        self.input_lang = input_lang
        self.output_lang = output_lang
        self.device = device
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        pair = self.pairs[idx]
        input_tensor = tensorFromSentence(self.input_lang, pair[0], self.device)
        target_tensor = tensorFromSentence(self.output_lang, pair[1], self.device)
        return input_tensor, target_tensor

# %% [markdown]
# ### 5. Implémentation des modèles récurrents

# %%
# RNN Simple
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=2):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x, hidden):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded, hidden)
        output = self.fc(output)
        return output, hidden
    
    def initHidden(self, batch_size, device):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)

# LSTM
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x, hidden, cell):
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        output = self.fc(output)
        return output, hidden, cell
    
    def initHidden(self, batch_size, device):
        return (torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device),
                torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device))

# GRU
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=2):
        super(GRUModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x, hidden):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded, hidden)
        output = self.fc(output)
        return output, hidden
    
    def initHidden(self, batch_size, device):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)

# %% [markdown]
# ### 6. Architecture Seq2Seq avec Encodeur-Décodeur

# %%
class Encoder(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=2, dropout=0.5):
        super(Encoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_size, hidden_size, num_layers=2, dropout=0.5):
        super(Decoder, self).__init__()
        self.output_size = output_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, hidden, cell):
        x = x.unsqueeze(1)
        embedded = self.dropout(self.embedding(x))
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        output = self.fc(output.squeeze(1))
        return output, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    
    def forward(self, source, target, teacher_forcing_ratio=0.5):
        batch_size = source.size(0)
        target_len = target.size(1)
        target_vocab_size = self.decoder.output_size
        
        outputs = torch.zeros(batch_size, target_len, target_vocab_size).to(self.device)
        
        _, hidden, cell = self.encoder(source)
        
        decoder_input = target[:, 0]
        
        for t in range(1, target_len):
            output, hidden, cell = self.decoder(decoder_input, hidden, cell)
            outputs[:, t] = output
            
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            decoder_input = target[:, t] if teacher_force else top1
        
        return outputs

# %% [markdown]
# ### 7. Fonctions d'entraînement avec gradient clipping

# %%
def train_epoch_seq2seq(model, dataloader, optimizer, criterion, device, teacher_forcing_ratio=0.5, clip=1.0):
    model.train()
    total_loss = 0
    
    for batch_idx, (input_tensor, target_tensor) in enumerate(dataloader):
        input_tensor = input_tensor.permute(1, 0, 2).squeeze(-1)
        target_tensor = target_tensor.permute(1, 0, 2).squeeze(-1)
        
        optimizer.zero_grad()
        
        output = model(input_tensor, target_tensor, teacher_forcing_ratio)
        
        loss = criterion(output.view(-1, output.shape[-1]), target_tensor.view(-1))
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def evaluate_random(model, input_lang, output_lang, pairs, device, n=5):
    model.eval()
    for i in range(n):
        pair = random.choice(pairs)
        input_tensor = tensorFromSentence(input_lang, pair[0], device).unsqueeze(0)
        
        with torch.no_grad():
            _, hidden, cell = model.encoder(input_tensor)
            
            decoder_input = torch.tensor([[SOS_token]], device=device)
            decoded_words = []
            
            for _ in range(MAX_LENGTH):
                output, hidden, cell = model.decoder(decoder_input, hidden, cell)
                topv, topi = output.topk(1)
                if topi.item() == EOS_token:
                    decoded_words.append('<EOS>')
                    break
                else:
                    if topi.item() < len(output_lang.index2word):
                        decoded_words.append(output_lang.index2word[topi.item()])
                    else:
                        decoded_words.append('<UNK>')
                decoder_input = topi
            
        print(f"Input: {pair[0]}")
        print(f"Target: {pair[1]}")
        print(f"Predicted: {' '.join(decoded_words[:-1])}")
        print("-" * 50)

# %% [markdown]
# ### 8. Comparaison RNN, LSTM, GRU

# %%
def compare_rnn_architectures():
    hidden_size = 256
    num_layers = 2
    learning_rate = 0.001
    num_epochs = 20
    
    architectures = {
        'RNN': SimpleRNN(input_lang.n_words, hidden_size, output_lang.n_words, num_layers),
        'LSTM': LSTMModel(input_lang.n_words, hidden_size, output_lang.n_words, num_layers),
        'GRU': GRUModel(input_lang.n_words, hidden_size, output_lang.n_words, num_layers)
    }
    
    results = {}
    
    for name, model in architectures.items():
        print(f"\n{'='*50}")
        print(f"Entraînement du {name}")
        print(f"{'='*50}")
        
        model = model.to(device)
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        criterion = nn.CrossEntropyLoss()
        
        losses = []
        for epoch in range(num_epochs):
            total_loss = 0
            model.train()
            
            for pair in pairs[:1000]:
                input_tensor = tensorFromSentence(input_lang, pair[0], device)
                target_tensor = tensorFromSentence(output_lang, pair[1], device)
                
                optimizer.zero_grad()
                
                if name == 'LSTM':
                    hidden, cell = model.initHidden(1, device)
                    loss = 0
                    for di in range(target_tensor.size(0)):
                        output, hidden, cell = model(input_tensor[di].view(1, -1), hidden, cell)
                        loss += criterion(output, target_tensor[di])
                else:
                    hidden = model.initHidden(1, device)
                    loss = 0
                    for di in range(target_tensor.size(0)):
                        output, hidden = model(input_tensor[di].view(1, -1), hidden)
                        loss += criterion(output, target_tensor[di])
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
                total_loss += loss.item()
            
            avg_loss = total_loss / len(pairs[:1000])
            losses.append(avg_loss)
            
            if (epoch + 1) % 5 == 0:
                print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")
        
        results[name] = losses
    
    # Visualisation
    plt.figure(figsize=(10, 6))
    for name, losses in results.items():
        plt.plot(losses, label=name, linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Comparaison RNN vs LSTM vs GRU')
    plt.legend()
    plt.grid(True)
    plt.savefig('rnn_comparison.png', dpi=150)
    plt.show()
    
    return results

# Exécution (prend du temps, optionnel)
# compare_results = compare_rnn_architectures()

# %% [markdown]
# ### 9. Entraînement du Seq2Seq complet

# %%
def train_seq2seq():
    hidden_size = 256
    num_layers = 2
    learning_rate = 0.001
    batch_size = 32
    num_epochs = 30
    
    encoder = Encoder(input_lang.n_words, hidden_size, num_layers).to(device)
    decoder = Decoder(output_lang.n_words, hidden_size, num_layers).to(device)
    model = Seq2Seq(encoder, decoder, device).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss(ignore_index=UNK_token)
    
    # Préparation du dataloader
    dataset = TranslationDataset(pairs[:5000], input_lang, output_lang, device)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=lambda x: x)
    
    train_losses = []
    
    for epoch in range(num_epochs):
        total_loss = 0
        model.train()
        
        for input_tensor, target_tensor in dataloader:
            input_tensor = input_tensor[0].to(device)
            target_tensor = target_tensor[0].to(device)
            
            input_tensor = input_tensor.permute(1, 0, 2).squeeze(-1)
            target_tensor = target_tensor.permute(1, 0, 2).squeeze(-1)
            
            optimizer.zero_grad()
            
            output = model(input_tensor, target_tensor, teacher_forcing_ratio=0.5)
            
            loss = criterion(output.view(-1, output.shape[-1]), target_tensor.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(dataloader)
        train_losses.append(avg_loss)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")
            evaluate_random(model, input_lang, output_lang, pairs, device, n=2)
    
    # Sauvegarde
    torch.save({
        'encoder_state': encoder.state_dict(),
        'decoder_state': decoder.state_dict(),
        'input_lang': input_lang,
        'output_lang': output_lang,
        'train_losses': train_losses
    }, 'seq2seq_model.pth')
    
    return model, train_losses

# %% [markdown]
# ### 10. Stratégies de décodage (Beam Search)

# %%
def greedy_decode(model, input_sentence, input_lang, output_lang, device):
    model.eval()
    input_tensor = tensorFromSentence(input_lang, input_sentence, device).unsqueeze(0)
    
    with torch.no_grad():
        _, hidden, cell = model.encoder(input_tensor)
        
        decoder_input = torch.tensor([[SOS_token]], device=device)
        decoded_words = []
        
        for _ in range(MAX_LENGTH):
            output, hidden, cell = model.decoder(decoder_input, hidden, cell)
            topv, topi = output.topk(1)
            
            if topi.item() == EOS_token:
                break
            
            if topi.item() < len(output_lang.index2word):
                decoded_words.append(output_lang.index2word[topi.item()])
            else:
                decoded_words.append('<UNK>')
            
            decoder_input = topi
    
    return ' '.join(decoded_words)

def beam_search_decode(model, input_sentence, input_lang, output_lang, device, beam_width=3, max_length=20):
    model.eval()
    input_tensor = tensorFromSentence(input_lang, input_sentence, device).unsqueeze(0)
    
    with torch.no_grad():
        _, hidden, cell = model.encoder(input_tensor)
        
        # Initialisation du beam
        sequences = [[[SOS_token], 0.0, hidden, cell]]
        
        for _ in range(max_length):
            all_candidates = []
            
            for seq, score, hidden, cell in sequences:
                if seq[-1] == EOS_token:
                    all_candidates.append((seq, score, hidden, cell))
                    continue
                
                decoder_input = torch.tensor([[seq[-1]]], device=device)
                output, new_hidden, new_cell = model.decoder(decoder_input, hidden, cell)
                
                log_probs = torch.log_softmax(output, dim=-1)
                top_log_probs, top_indices = log_probs.topk(beam_width)
                
                for i in range(beam_width):
                    next_token = top_indices[0][i].item()
                    next_score = score + top_log_probs[0][i].item()
                    all_candidates.append((seq + [next_token], next_score, new_hidden, new_cell))
            
            # Garder les beam_width meilleures séquences
            sequences = sorted(all_candidates, key=lambda x: x[1], reverse=True)[:beam_width]
            
            # Si toutes les séquences finissent par EOS
            if all(seq[-1] == EOS_token for seq, _, _, _ in sequences):
                break
        
        # Meilleure séquence
        best_seq = sequences[0][0]
        
        # Conversion en mots
        decoded_words = []
        for token in best_seq[1:]:  # Skip SOS
            if token == EOS_token:
                break
            if token < len(output_lang.index2word):
                decoded_words.append(output_lang.index2word[token])
            else:
                decoded_words.append('<UNK>')
        
        return ' '.join(decoded_words)

def compare_decoding_strategies(model, input_lang, output_lang, pairs, device, n=5):
    print("\n" + "="*50)
    print("COMPARAISON DES STRATÉGIES DE DÉCODAGE")
    print("="*50)
    
    for i in range(n):
        pair = random.choice(pairs)
        print(f"\nPhrase source: {pair[0]}")
        print(f"Traduction attendue: {pair[1]}")
        
        greedy_output = greedy_decode(model, pair[0], input_lang, output_lang, device)
        print(f"Décodage glouton: {greedy_output}")
        
        beam_output = beam_search_decode(model, pair[0], input_lang, output_lang, device, beam_width=3)
        print(f"Beam search (k=3): {beam_output}")
        print("-" * 40)

# %% [markdown]
# ### 11. Évaluation avec métrique BLEU

# %%
!pip install nltk -q
import nltk
nltk.download('punkt', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def compute_bleu_scores(model, input_lang, output_lang, pairs, device, n=100):
    model.eval()
    bleu_scores = []
    
    smoothie = SmoothingFunction().method4
    
    for pair in random.sample(pairs, min(n, len(pairs))):
        reference = pair[1].split()
        prediction = greedy_decode(model, pair[0], input_lang, output_lang, device).split()
        
        if len(prediction) > 0:
            bleu = sentence_bleu([reference], prediction, smoothing_function=smoothie)
            bleu_scores.append(bleu)
    
    return np.mean(bleu_scores), np.std(bleu_scores)

# %%
print("\n" + "="*50)
print("ÉVALUATION FINALE Seq2Seq")
print("="*50)

# Si le modèle est déjà entraîné, chargez-le
try:
    checkpoint = torch.load('seq2seq_model.pth', map_location=device)
    encoder = Encoder(input_lang.n_words, 256, 2).to(device)
    decoder = Decoder(output_lang.n_words, 256, 2).to(device)
    model = Seq2Seq(encoder, decoder, device).to(device)
    encoder.load_state_dict(checkpoint['encoder_state'])
    decoder.load_state_dict(checkpoint['decoder_state'])
    print("Modèle chargé avec succès!")
    
    # Calcul du BLEU
    mean_bleu, std_bleu = compute_bleu_scores(model, input_lang, output_lang, pairs, device, n=200)
    print(f"\nBLEU Score moyen: {mean_bleu:.4f} (+/- {std_bleu:.4f})")
    
    # Comparaison des stratégies
    compare_decoding_strategies(model, input_lang, output_lang, pairs, device, n=3)
    
except FileNotFoundError:
    print("Aucun modèle entraîné trouvé. Exécutez d'abord la cellule d'entraînement.")

# %% [markdown]
# ### 12. Visualisation des pertes d'entraînement

# %%
# Si vous avez entraîné le modèle, tracez les pertes
try:
    checkpoint = torch.load('seq2seq_model.pth', map_location=device)
    train_losses = checkpoint['train_losses']
    
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Courbe d\'apprentissage - Seq2Seq Translation')
    plt.grid(True)
    plt.savefig('seq2seq_training_loss.png', dpi=150)
    plt.show()
except:
    print("Aucune courbe de perte disponible.")
    # À la fin de votre notebook Partie 3
torch.save({
    'encoder_state': encoder.state_dict(),
    'decoder_state': decoder.state_dict(),
    'input_lang': input_lang,
    'output_lang': output_lang
}, 'seq2seq_model.pth')

In [ ]:
# ==================== TEST DU MODÈLE ====================
def evaluate(encoder, decoder, sentence, max_length=10):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        input_length = input_tensor.size(0)
        encoder_hidden = encoder.initHidden()
        
        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
        
        decoder_input = torch.tensor([[SOS_token]], device=device)
        decoder_hidden = encoder_hidden
        
        decoded_words = []
        
        for di in range(max_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = decoder_output.topk(1)
            if topi.item() == EOS_token:
                break
            else:
                if topi.item() < len(output_lang.index2word):
                    decoded_words.append(output_lang.index2word[topi.item()])
                else:
                    decoded_words.append('?')
            decoder_input = topi
        
        return ' '.join(decoded_words)

print("\n" + "=" * 50)
print("TESTS DE TRADUCTION")
print("=" * 50)

test_phrases = ["bonjour", "merci", "au revoir", "comment allez vous", "je vais bien"]

for phrase in test_phrases:
    # Normaliser la phrase d'entrée
    from data_utils import normalizeString
    normalized = normalizeString(phrase)
    
    # Vérifier si la phrase existe dans les données
    matching_pairs = [p for p in pairs if p[0] == normalized]
    
    translation = evaluate(encoder, decoder, normalized)
    print(f"\nFrançais: {phrase}")
    print(f"Traduction: {translation}")
    
    if matching_pairs:
        print(f"Attendu: {matching_pairs[0][1]}")

print("\n✅ Tests terminés !")